# 04 — Diagnostics du modèle bayésien

**Auteur :** Samir EL AISSAOUY  
**Objectif :** Valider la convergence MCMC et la qualité du modèle bayésien

Contenu :
1. Diagnostics MCMC (R-hat, ESS, divergences)
2. Trace plots
3. Posterior Predictive Check (PPC)
4. Walk-Forward Validation
5. Distributions postérieures des paramètres
6. Rapport de validation complet


## 1. Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader           import load_market_data, split_train_test
from src.models.bayesian_mmm        import BayesianMMM
from src.training.model_diagnostics import (
    check_rhat, check_ess, check_divergences,
    compute_loo, compute_ppc_metrics,
    full_diagnostics_report,
)
from src.evaluation.model_validation import (
    posterior_predictive_check,
    walk_forward_validation,
    full_validation_report,
)
from src.evaluation.metrics         import compute_all_metrics, print_metrics_report
from src.utils.visualization        import (
    plot_actual_vs_predicted,
    plot_posterior_distributions,
)

MARKET = 'FR'
print(f' Imports OK — Diagnostics marché {MARKET}')

## 2. Entraînement du modèle

In [ ]:
df = load_market_data(MARKET)
df_train, df_test = split_train_test(df, test_ratio=0.2)

print(f'Train : {len(df_train)} semaines')
print(f'Test  : {len(df_test)} semaines')

model = BayesianMMM(market=MARKET)
model.fit(df_train, draws=1000, tune=1000, chains=4, random_seed=42)

print(f'\n{repr(model)}')
print(f'PyMC disponible : {model.idata is not None}')

## 3. Diagnostics MCMC — R-hat, ESS, Divergences

In [ ]:
if model.idata is not None:
    print('── R-hat (Gelman-Rubin) ──────────────────────────')
    rhat_issues = check_rhat(model.idata, threshold=1.01)
    print(f'Paramètres R-hat ≥ 1.01 : {len(rhat_issues)}')

    print('\n── ESS (Effective Sample Size) ───────────────────')
    ess_issues = check_ess(model.idata, min_ess=400)
    print(f'Paramètres ESS < 400    : {len(ess_issues)}')

    print('\n── Divergences HMC ───────────────────────────────')
    div = check_divergences(model.idata)
    print(f'Divergences             : {div["n_divergences"]} ({div["pct_divergences"]:.2f}%)')

    print('\n── LOO-CV ────────────────────────────────────────')
    loo = compute_loo(model.idata)
    if loo:
        print(f'elpd_loo : {loo["elpd_loo"]:.2f} (±{loo["se"]:.2f})')
        print(f'p_loo    : {loo["p_loo"]:.2f}')

    convergence_ok = (
        len(rhat_issues) == 0 and
        len(ess_issues)  == 0 and
        div['n_divergences'] == 0
    )
    print(f'\n→ Convergence globale : {"✅ OK" if convergence_ok else "⚠️  Problèmes détectés"}')
else:
    print('⚠️  Mode OLS — diagnostics MCMC non disponibles')
    print(model.diagnostics()['message'])

## 4. Summary arviz — tableau complet

In [ ]:
if model.idata is not None:
    import arviz as az
    summary = az.summary(model.idata, round_to=4)
    print('Résumé postérieur (mean, sd, HDI 94%, R-hat, ESS) :')
    display(summary)
else:
    print('⚠️  Summary disponible uniquement avec PyMC')

## 5. Trace plots — convergence visuelle

In [ ]:
if model.idata is not None:
    import arviz as az
    params = ['beta_tv', 'beta_facebook', 'beta_search', 'beta_ooh', 'beta_print']
    available = [p for p in params if p in model.idata.posterior]
    if available:
        az.plot_trace(
            model.idata,
            var_names=available,
            figsize=(12, len(available) * 2)
        )
        plt.suptitle(f'Trace plots — {MARKET}', fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.show()
else:
    print('⚠️  Trace plots disponibles uniquement avec PyMC')

## 6. Distributions postérieures des paramètres

In [ ]:
if model.idata is not None:
    fig = plot_posterior_distributions(
        model.idata,
        params=['beta_tv', 'beta_facebook', 'beta_search', 'beta_ooh', 'beta_print']
    )
    plt.show()
else:
    print('⚠️  Distributions postérieures disponibles uniquement avec PyMC')

## 7. Posterior Predictive Check (PPC)

In [ ]:
if model.idata is not None:
    ppc = posterior_predictive_check(model.idata, df_test, n_samples=200)

    print('── Résultats PPC ─────────────────────────────────')
    print(f'Coverage HDI 94%  : {ppc.get("coverage_94", "N/A"):.1f}%  (objectif > 80%)')
    print(f'Moyenne observée  : €{ppc.get("mean_observed", 0):,.0f}')
    print(f'Moyenne PPC       : €{ppc.get("mean_ppc", 0):,.0f}')
    print(f'p-value mean      : {ppc.get("p_value_mean", "N/A"):.3f}  (idéal : 0.4–0.6)')
    print(f'p-value std       : {ppc.get("p_value_std",  "N/A"):.3f}')
    print(f'\nÉvaluation        : {ppc.get("assessment", "N/A")}')

    # Visualisation PPC
    if 'ppc_samples' in ppc:
        samples  = ppc['ppc_samples']
        y_obs    = df_test['revenue'].values
        lower    = np.percentile(samples, 3,  axis=0)
        upper    = np.percentile(samples, 97, axis=0)
        mean_ppc = samples.mean(axis=0)

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.fill_between(range(len(y_obs)), lower, upper,
                        alpha=0.2, color='#2196F3', label='HDI 94%')
        ax.plot(y_obs,    color='black',   linewidth=1.5, label='Observé')
        ax.plot(mean_ppc, color='#2196F3', linewidth=2,   linestyle='--', label='Moyenne PPC')
        ax.set_title(f'Posterior Predictive Check — {MARKET}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Semaine (test set)')
        ax.set_ylabel('Revenue (€)')
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print('⚠️  PPC disponible uniquement avec PyMC')

## 8. Walk-Forward Validation (5 splits)

In [ ]:
print('⏳ Walk-Forward Validation en cours...')
wf_results = walk_forward_validation(
    df, BayesianMMM, config={}, n_splits=5, min_train_size=100
)

print('\nRésultats Walk-Forward (5 splits temporels) :')
print(wf_results[['split', 'n_train', 'n_test', 'r2', 'mape', 'nrmse']].to_string(index=False))
print(f'\nR²   moyen : {wf_results["r2"].mean():.3f}  (std={wf_results["r2"].std():.3f})')
print(f'MAPE moyen : {wf_results["mape"].mean():.1f}% (std={wf_results["mape"].std():.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# R²
axes[0].plot(wf_results['split'], wf_results['r2'], 'o-', color='#2196F3', linewidth=2, markersize=8)
axes[0].axhline(0.85, color='red', linestyle='--', linewidth=1.5, label='Objectif 0.85')
axes[0].fill_between(wf_results['split'],
                     wf_results['r2'] - wf_results['r2'].std(),
                     wf_results['r2'] + wf_results['r2'].std(),
                     alpha=0.15, color='#2196F3')
axes[0].set_title('R² par split', fontweight='bold')
axes[0].set_xlabel('Split')
axes[0].set_ylim(0, 1.05)
axes[0].legend()
axes[0].grid(alpha=0.3)

# MAPE
axes[1].plot(wf_results['split'], wf_results['mape'], 'o-', color='#FF9800', linewidth=2, markersize=8)
axes[1].axhline(15, color='red', linestyle='--', linewidth=1.5, label='Objectif 15%')
axes[1].set_title('MAPE par split (%)', fontweight='bold')
axes[1].set_xlabel('Split')
axes[1].legend()
axes[1].grid(alpha=0.3)

# NRMSE
axes[2].plot(wf_results['split'], wf_results['nrmse'], 'o-', color='#9C27B0', linewidth=2, markersize=8)
axes[2].axhline(0.15, color='red', linestyle='--', linewidth=1.5, label='Objectif 0.15')
axes[2].set_title('NRMSE par split', fontweight='bold')
axes[2].set_xlabel('Split')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle(f'Walk-Forward Validation — {MARKET}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Rapport complet de validation

In [ ]:
y_pred = model.predict(df_test)
report = full_validation_report(
    idata=model.idata,
    df=df_test,
    y_pred=y_pred,
    market=MARKET,
)

print('\n' + '=' * 60)
print(f'  RAPPORT FINAL — {MARKET}')
print('=' * 60)
print(f'  {report["summary"]}')
print('─' * 60)
m = report['metrics']
print(f'  R²    : {m["r2"]:.4f}   (objectif ≥ 0.85)')
print(f'  MAPE  : {m["mape"]:.2f}%  (objectif ≤ 15%)')
print(f'  NRMSE : {m["nrmse"]:.4f}  (objectif ≤ 0.15)')
print(f'  RMSE  : €{m["rmse"]:,.0f}')
if 'convergence_ok' in report:
    print(f'  Convergence MCMC : {"✅" if report["convergence_ok"] else "⚠️"}')
print('=' * 60)

fig = plot_actual_vs_predicted(df_test, y_pred, market=MARKET)
plt.show()